# Train All Models on Colab GPU

**Before you run:** Runtime → Change runtime type → **GPU** (T4 is fine).

Training order (fastest first):
1. Mount Drive + clone repo
2. **XGBoost** — trains in seconds, no GPU needed
3. **Per-Player** — trains in seconds, no GPU needed
4. **LSTM** — ~2 min on T4
5. **CNN** — ~5 min on T4
6. **Fusion** — ~3 min on T4 (requires LSTM + CNN checkpoints)
7. Evaluate all 5 models + save checkpoints to Drive

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
candidates = [
    '/content/drive/MyDrive/podz-liability-fund',
    '/content/drive/MyDrive/Podz-Liability-Fund',
    '/content/drive/Shareddrives/podz-liability-fund',
]
GDRIVE = next((p for p in candidates if os.path.isdir(p)), None)
if GDRIVE is None:
    raise FileNotFoundError('Drive folder podz-liability-fund not found. Add a shortcut to My Drive.')
os.environ['PODZ_DATA_DIR'] = GDRIVE
print('Using:', GDRIVE)
print('Contents:', sorted(os.listdir(GDRIVE)))

## 2. Clone repo + install deps

In [ ]:
import os, sys, subprocess
REPO_DIR = '/content/Podz-Liability-Fund'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Nathan-Dela-Pena/Podz-Liability-Fund.git',
                    REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('repo:', REPO_DIR)

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Runtime → Change runtime type → GPU')

## 3. Verify train/val/test sequences have prop-line features

The repo already contains `data/processed/{train,val,test}.csv` with:
- **`PROP_LINE_Z`** — prop line in z-score units
- **`LINE_VS_RECENT`** — `(line - recent_mean_pts) / per_player_std`

No regeneration needed — just verify the columns are present before training.

In [ ]:
import pandas as pd

required = ["PROP_LINE_Z", "LINE_VS_RECENT"]
df = pd.read_csv("data/processed/train.csv")
missing = [c for c in required if c not in df.columns]
if missing:
    raise RuntimeError("train.csv is missing columns: " + str(missing) + ". Pull the latest repo: git pull --ff-only")
print("OK — prop-line columns present in train.csv")
print("Columns:", list(df.columns)[:6], "... (len", len(df.columns), ")")
print(df[["PROP_LINE", "PROP_LINE_Z", "LINE_VS_RECENT", "LABEL"]].head())

## 4. Train XGBoost (gradient boosting on flat rolling features)

Gradient boosting on small tabular data typically outperforms LSTMs.
Uses the same window features as the LSTM — no new data needed.

In [ ]:
!pip install -q xgboost
from src.models.xgb_model import train_xgb
xgb_model = train_xgb()

## 5. Train Per-Player models (player-specific logistic regression)

Trains a separate logistic regression per player so each model specializes
on that player's individual scoring patterns. Falls back to a global model
for players with fewer than 15 training samples.

In [ ]:
from src.models.per_player import train_per_player
per_player_bundle = train_per_player()

## 6. Train LSTM (with prop-line static features)

The LSTM now consumes the rolling-window stats AND the static prop-line features through a small MLP merged into the FC head. Watch `val_auc` — should land well above the previous ~0.51.

In [ ]:
from src.models.lstm import train_lstm
lstm_model = train_lstm(device='cuda')

## 7. Train CNN (player-identity encoder)

Trains the visual + pose encoder via a player-identity auxiliary task. The 192-d hidden state is what the fusion model consumes.

In [ ]:
from src.models.cnn import train_cnn
cnn_model = train_cnn(device='cuda')

## 8. Train Fusion (LSTM + CNN + prop-line, end-to-end)

Loads the freshly trained `lstm_best.pt` and `cnn_best.pt`, then fine-tunes end-to-end. The classifier head also receives the prop-line static features directly so cross-modal mixing can condition on them.

In [ ]:
from src.models.fusion import train_fusion
fusion_model = train_fusion(device='cuda')

## 9. Evaluate on the test set

In [ ]:
!python scripts/evaluate.py

## 10. Save checkpoints to Drive

Colab wipes `/content` on disconnect — copy the `.pt` and `.pkl` files to Drive.

In [ ]:
import shutil, pathlib
src_dir = pathlib.Path("checkpoints")
dst_dir = pathlib.Path(os.environ["PODZ_DATA_DIR"]) / "checkpoints"
dst_dir.mkdir(parents=True, exist_ok=True)
for f in sorted(list(src_dir.glob("*.pt")) + list(src_dir.glob("*.pkl"))):
    shutil.copy2(f, dst_dir / f.name)
    print(f"copied {f.name} ({f.stat().st_size / 1e6:.1f} MB)  ->  {dst_dir / f.name}")